# HYDE
## Get started
## Prepare the data
<img src="hyde.png">
我们使用 Langchain WebBaseLoader 从博客源加载文档，并通过 RecursiveCharacterTextSplitter 将其拆分为多个片段。

In [1]:
import os

CUSTOM_CACHE = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')
os.environ['TORCH_HOME'] = CUSTOM_CACHE

In [2]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a WebBaseLoader instance to load documents from web sources
loader = WebBaseLoader(
    web_path=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content","post-title","post-header")
        ),# 只解析 HTML 中符合特定条件的部分
    )
)

# Load documents from web sources using the loader
documents=loader.load()

# Initialize a RecursiveCharacterTextSplitter for splitting text into chunks
text_spliiter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=0)

# Split the documents into chunks using the text_splitter
docs=text_spliiter.split_documents(documents)

# Inspect
docs[1]

C:\Users\Administrator\AppData\Local\Temp\ipykernel_3708\2095123501.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Short-term memory: I would consider all the in-context learning (See Prompt Engineering) as utilizing short-term memory of the model to learn.\nLong-term memory: This provides the agent with the capability to retain and recall (infinite) information over extended periods, often by leveraging an external vector store and fast retrieval.\n\n\nTool use\n\nThe agent learns to call external APIs for extra information that is missing from the model weights (often hard to change after pre-training), including current information, code execution capability, access to proprietary information sources and more.\n\n\n\n\n\nOverview of a LLM-powered autonomous agent system.')

## Build the chain

We load the docs into milvus vectorstore, and build a milvus retriever.

In [4]:
from rag_utils.vanilla import vectorstore
vectorstore.add_documents(docs)
retriever=vectorstore.as_retriever()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Build the vanilla RAG chain.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from rag_utils.vanilla import format_docs, rag_prompt, llm

## 流水线接力（管道符 | 链）
    # 第一站（数据准备）：你输入问题 "What is ..."，经过车间，输出一个字典 {"context": "拼接好的参考资料", "question": "你的原始问题"}。
    # 第二站（拼提示词 rag_prompt）：把上面字典里的 context 和 question 填进你定义的 PROMPT_TEMPLATE 模版里（替换掉 {context} 和 {question} 占位符），生成一段完整的、包含背景资料的“指令文本”。
    # 第三站（交给大模型 llm）：把填好的提示词发给 DeepSeek 大模型，模型开始“阅读”资料并思考回答。
    # 第四站（提取纯文本 StrOutputParser）：大模型返回的是一个包含各种元数据的对象（比如用了多少 token），StrOutputParser 只把最终的答案文字提取出来，去掉包装。
vanilla_rag_chain=(
    {"context":retriever|format_docs, "question":RunnablePassthrough()}
    |rag_prompt
    |llm
    |StrOutputParser()
)

Build a **hyde chain**.

用AI先根据你的问题“瞎编”一个假答案，然后用这个假答案去库里找相似的文档。

In [ ]:
from rag_utils.hyde import HydeRetriever

hyde_retriever=HydeRetriever.from_vectorstore(vectorstore)

hyde_chain=(
    {"context":hyde_retriever|format_docs, "question":RunnablePassthrough()}
    |rag_prompt
    |llm
    |StrOutputParser()
)


## Test the chain

In [ ]:
# query = 哪些向量近似搜索算法适用于向量存储
query = "which vector approximate searching algorithms work in a vector store"

vanilla_result=vanilla_rag_chain.invoke(query)
hyde_result=hyde_chain.invoke(query)
print(f"\n[vanilla_result]:\n{vanilla_result}\n\n[hyde_result]:\n{hyde_result}")

在 [hyde_result] 中，它使用了与真实数据“HNSW”匹配的结果，而该结果并未出现在 [vanilla_result] 中。

让我们深入分析检索到的结果，找出原因。

hyde_retriever 使用 HNSW 提取了文档，而 vanilla_retriever 则没有。这是因为 HyDE 生成了包含多种向量近似搜索算法（如 HNSW）的虚假文档，从而使得检索器的结果更加准确。